In [ ]:
pip install sentence-transformers faiss-cpu transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM # text to Embedding
import numpy as np
import faiss # vector DB

This is your dataset or knowledge source.

You can replace these with text from PDFs, websites, etc.



In [ ]:
documents = [
    "The Eiffel Tower is in Paris, France.",
    "The Great Wall of China is visible from space.",
    "Cristiano Ronaldo is a famous Portuguese footballer.",
    "Python is a widely used programming language for AI.",
    "OpenAI developed the GPT series of language models."
]

 ### Step 2: Load Embedding Model

 Loads a pretrained model to convert text → vectors (embeddings).

MiniLM is small, fast, and good for semantic similarity.

In [ ]:
embedder=SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Step 3: Create Embeddings for Documents

Converts each document into a fixed-size numeric vector (shape: 384).


This is needed so we can compare them mathematically.


In [ ]:
doc_embedding=embedder.encode(documents)

### creating the vector db
**Step 4: Build FAISS Index**

IndexFlatL2 uses L2 (Euclidean) distance to find similar vectors.

Adds your document vectors into the search index.

In [ ]:
dimension=doc_embedding.shape[1] # to get the shape of dataset to know the column dimensions
index=faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embedding))

### Step 5: User Query
Converts the question into an embedding to compare with documents.



In [ ]:
query="Who cerated models?"
query_embedding=embedder.encode([query])

# Step 6: Search Top-k Relevant Documents

Searches the vector DB to find top 2 similar documents.

retrieved_docs will be passed to the LLM as context.



In [ ]:
top_k = 2
D, I = index.search(np.array(query_embedding), top_k)
retrieved_docs = [documents[i] for i in I[0]]


# Step 7: Create Prompt for LLM

Combine context (retrieved docs) and the query into one prompt.


This prompt guides the LLM to answer using only relevant info.


✅ Let's say this is your query:



# query = "Who created GPT models?"

🟡 After similarity search (Step 6), these 2 documents were retrieved:



#  retrieved_docs = [
    "OpenAI developed the GPT series of language models.",
    "Python is a widely used programming language for AI."
 ]
 ## New Section
## ✅ 🧾 Final Prompt sent to the LLM:

Context: OpenAI developed the GPT series of language models. Python is a widely used programming language for AI.

Question: Who created GPT models?

Answer:



In this step the we get the retrived data(relevant _data) for query from rag


 Does the LLM add extra info, or only use what’s in the Vector DB?


✅ Answer:


🔹 LLM can add extra information if it knows it from training


🔹 But mostly it tries to stick to the context you give from the Vector DB


🔹 It’s not guaranteed to only use Vector DB — it’s still a language model!

1. User asks question ➝
2. Convert to vector ➝
3. Search in FAISS ➝
4. Retrieve relevant docs ➝
5. Build prompt (context + question) ➝
6. 🔥 LLM uses prompt to generate answer


In [ ]:
# 7. Build prompt for LLM
context = " ".join(retrieved_docs)
prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"

# Step 8: Load the LLM (FLAN-T5)

Loads FLAN-T5, a small but effective text-generation model.

You can replace it with a bigger LLM if needed (e.g., LLaMA, Mistral).


### the content get based on retrieved data will divided into token see step 7 example

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

# 9. Generate answer

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
outputs = model.generate(**inputs, max_length=100)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print("Query:", query)
print("Answer:", answer)

Query: Who cerated models?
Answer: OpenAI


# see notes cr7 for better explanation
 in this we are getting the data from rag where llm doesnt have information about where we use rag to get:

 for example : who is manivarma?


 based on llm  we does nt get it through rag  we add our personal information and through llm we generate the information

📦 RAG = Retrieval + Generation

Component	Description	Example Tool

🔍 Retriever:	Finds relevant documents based on user query	FAISS, Chroma, Weaviate(vector DB)


🧠 Generator:	Uses the retrieved text to generate an answer	FLAN-T5, GPT, BERT, Mistral( FROM VECTOR DB TO HUMAN Understanding Language(llm))
#

🧠 Flow Recap:

1. User asks question ➝
2. Convert to vector ➝
3. Search in FAISS ➝(vector database)
4. Retrieve relevant docs ➝
5. Build prompt (context + question) ➝
6. 🔥 LLM uses prompt to generate answer
